# XP Exercises: Flower Classification using CNN

This is a guided notebook for the exercises on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts that unlock intuition or transfer to other ML topics.


## What you will learn
- Building a CNN for multi class image classification
- Data loading and preprocessing with `image_dataset_from_directory`
- Image visualization techniques
- Model architecture design, compilation, and training
- Evaluating model performance with accuracy and loss plots


## What you will create
A CNN model that classifies 14 flower species.
All parts form one continuous exercise. Work through them sequentially.


## Dataset
**As stated in the exercises**  
Flower classification with 14 classes. Images are organized in class folders. A training and validation split may be provided. Images are resized to 256x256 in this notebook.

**PREFILLED info**  
This notebook expects the provided zip file to be available. The code below extracts it and locates the dataset root automatically.


In [ ]:
# PREFILLED: just execute
import os, sys, zipfile, shutil, glob, math, json, random
from pathlib import Path

DATA_ZIP = Path("./Flower Classification.zip")
EXTRACT_DIR = Path("./data/flower_data")

# Clean extract dir if re-running
if EXTRACT_DIR.exists():
    pass  # avoid deleting in case you added files; delete manually if needed
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract if a zip is present and not already extracted
if DATA_ZIP.exists():
    # Heuristically decide to extract once
    marker = EXTRACT_DIR / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text("ok")
        print("Extracted:", DATA_ZIP.name, "->", EXTRACT_DIR)
    else:
        print("Already extracted. Skipping.")
else:
    print("Zip file not found at", DATA_ZIP)

# Find candidate dataset roots: a dir with >= 10 subdirs assumed as classes, or contains train/val
def list_dirs(p):
    return [d for d in Path(p).iterdir() if d.is_dir()]

candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    if len([d for d in Path(root).iterdir() if Path(d).is_dir()]) >= 10:
        candidates.append(Path(root))
    if "train" in [d.name.lower() for d in list_dirs(root)] and "val" in [d.name.lower() for d in list_dirs(root)]:
        candidates.append(Path(root))

candidates = sorted(set(candidates))
print("Candidate dataset roots:", [str(c) for c in candidates][:5])

## Part 1. Data exploration and visualization

**As stated in the exercises**  
Load the dataset using `image_dataset_from_directory`. Print number of images per class. Modify `visualize_images` to show a 3x3 grid for each class with the class name as the grid title. Analyze challenges you anticipate when classifying the flowers such as similar colors or shapes and intra class variation.


**Guidance**  
If a `train` or `val` folder exists, use them. Otherwise create a split from a single root with `validation_split` and `subset`. Images are resized to 256x256 RGB.


> **IMPORTANT:** we fix a low resolution for images in IMG_SIZE=(32,32) for faster training, however you can change it if you want to test out other resolutions

In [ ]:
# PREFILLED: just execute
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers

IMG_SIZE = (32, 32)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def detect_layout(root: Path):
    root = Path(root)
    sub = [d.name.lower() for d in root.iterdir() if d.is_dir()]
    if "train" in sub and "val" in sub:
        return "provided_split", root
    return "single_root", root

# Choose a root
if 'candidates' in globals() and len(candidates) > 0:
    DS_ROOT = candidates[0]
else:
    DS_ROOT = EXTRACT_DIR  # fallback

layout, base = detect_layout(DS_ROOT)
print("Layout:", layout, "Base:", base)

In [ ]:
# PREFILLED: just execute
if layout == "provided_split":
    train_dir = next((p for p in base.iterdir() if p.name.lower()=="train"))
    val_dir   = next((p for p in base.iterdir() if p.name.lower()=="val"))
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", num_classes, class_names)

# Cache and prefetch
def prepare(ds):
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds)
val_ds = prepare(val_ds)

In [ ]:
# PREFILLED: just execute — count images per class by scanning directory
from collections import Counter
import os

def count_images_per_class(root):
    counts = {}
    for cls in class_names:
        # find folder named like cls at any depth under base
        matches = list(Path(root).rglob(cls))
        if matches:
            folder = matches[0]
            img_count = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".gif"})
            counts[cls] = img_count
        else:
            counts[cls] = None
    return counts

# Derive counting root WITHOUT overwriting the global `base`
count_root = train_dir if layout == "provided_split" else base
counts = count_images_per_class(count_root)
counts

In [ ]:
# To-Do: visualize the class distribution as a bar chart
valid_counts = {k: v for k, v in counts.items() if v is not None}
classes_sorted = sorted(valid_counts, key=valid_counts.get, reverse=True)
values_sorted  = [valid_counts[c] for c in classes_sorted]

plt.figure(figsize=(12, 4))
bars = plt.bar(classes_sorted, values_sorted,
               color=plt.cm.tab20.colors[:len(classes_sorted)],
               edgecolor='black', linewidth=0.6)
for bar, val in zip(bars, values_sorted):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(val), ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.ylabel('Number of images')
plt.title('Images per Flower Class (Training Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Total training images: {sum(valid_counts.values())}")

In [ ]:
# To-Do: implement visualize_images to display a 3x3 grid for each class
def visualize_images(dataset, class_names, per_class=9):
    """
    Display a 3×3 grid of images for every class in class_names.
    The figure suptitle shows the class name.
    """
    # ── Collect up to per_class images for each label ────────────────────
    buckets = {i: [] for i in range(len(class_names))}
    needed  = set(range(len(class_names)))  # labels we still need

    for images, labels in dataset.unbatch():
        lbl = int(labels.numpy())
        if lbl in needed and len(buckets[lbl]) < per_class:
            buckets[lbl].append(images.numpy().astype('uint8'))
        # stop early when all buckets are full
        if all(len(buckets[i]) >= per_class for i in range(len(class_names))):
            break

    # ── Plot a 3×3 grid per class ────────────────────────────────────────
    cols = 3
    rows = math.ceil(per_class / cols)   # = 3 for per_class=9

    for label_idx, class_name in enumerate(class_names):
        imgs = buckets[label_idx]
        if not imgs:
            print(f"No images found for class: {class_name}")
            continue

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
        fig.suptitle(class_name, fontsize=13, fontweight='bold', y=1.01)

        for idx, ax in enumerate(axes.flat):
            if idx < len(imgs):
                ax.imshow(imgs[idx])
            ax.axis('off')

        plt.tight_layout()
        plt.show()

# Run on training set
visualize_images(train_ds, class_names, per_class=9)

The `visualize_images` function above iterates the training dataset and fills a 3×3 grid for each of the 14 flower classes. Run the cell above to verify that images and class titles render correctly.


**Classification challenges — analysis:**

Classifying these 14 flower species presents several difficulties. First, **color overlap** is significant: Calendula, Common Daisy, and Coreopsis all feature yellow petals, while Roses and Carnations share a similar pinkish-red palette, making color alone an unreliable discriminator. Second, **shape similarity** is common at low resolution — Dandelion seed heads and some Asters are both circular with fine radiating structures, and the network must learn subtle differences in petal arrangement. Third, **intra-class variation** is high: a single species like Rose can appear as tight buds, half-open blooms, or fully open flowers in many colors (red, white, yellow), so the model must capture underlying morphological features rather than surface appearance. Fourth, **background clutter and lighting variability** mean that the same flower photographed outdoors in bright sunlight vs. indoors under artificial light can look very different in pixel space. Finally, **class imbalance** — some species (e.g., Dandelion, Sunflower) typically have more images than others (e.g., Water Lily, Carnation) — can bias the model toward majority classes, inflating overall accuracy while under-performing on rare classes. These factors together mean that simple pixel-level patterns are insufficient; the network needs hierarchical feature extraction and regularization to generalize well.


**Learning point**  
Vision models learn features from texture, color, and shape. Dataset bias and imbalance can dominate results without careful preprocessing and evaluation.


## Part 2. Model architecture design

**As stated in the exercises**  
Start from the provided model. Experiment with the number of convolutional layers, filters, kernel sizes, max pooling layers. Try different dense layers and dropout. Consider Batch Normalization. Justify your architectural choices.


In [ ]:
# PREFILLED: just execute — baseline model scaffold
from tensorflow.keras import models

def build_baseline(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),  # safety if datasets were not normalized
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline(num_classes)
baseline.summary()

In [ ]:
# To-Do: create an improved architecture variant
def build_variant(num_classes):
    """
    Improved CNN with:
    - Progressively increasing filters: 32 -> 64 -> 128 -> 256
    - BatchNormalization after every Conv2D for training stability
    - A 5x5 kernel in the first layer to capture larger patterns
    - Two Dense layers with higher dropout (0.4) to fight overfitting
    """
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),

        # ── Block 1: 5×5 kernel — wider receptive field from the start ───
        layers.Conv2D(32, kernel_size=5, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=2),

        # ── Block 2 ────────────────────────────────────────────────────
        layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=2),

        # ── Block 3 ────────────────────────────────────────────────────
        layers.Conv2D(128, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=2),

        # ── Block 4 (no pooling — feature map is already small) ─────────
        layers.Conv2D(256, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),

        # ── Classifier head ────────────────────────────────────────────
        layers.GlobalAveragePooling2D(),   # reduces spatial dims, fewer params than Flatten
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Variant')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_variant = build_variant(num_classes)
model_variant.summary()

**Architecture justification:**

The first convolutional layer uses a **5×5 kernel** instead of 3×3 to capture broader spatial patterns — petals and flower centers often span more than three pixels even at 32×32 resolution, so the larger receptive field improves early feature extraction. Filters are increased progressively from **32 → 64 → 128 → 256** following the standard CNN convention: early layers detect simple edges and textures, while deeper layers compose them into semantically richer representations such as petal arrangements and stamen shapes. **Batch Normalization** is placed after every Conv2D layer because it standardizes layer inputs to zero mean and unit variance, accelerating convergence and acting as a mild regularizer — this is especially important here since the pixel values of flowers vary widely across lighting conditions. **GlobalAveragePooling2D** replaces a naive Flatten layer to reduce the spatial feature maps to a single vector per filter, dramatically cutting the parameter count in the dense head and further reducing overfitting risk. Finally, two **Dropout** layers (rates 0.4 and 0.3) are stacked before the output to prevent co-adaptation of neurons in the fully connected head, which is the part most prone to memorizing training examples when the dataset is moderate in size.


## Part 3. Hyperparameter tuning

**As stated in the exercises**  
Experiment with optimizers, learning rate, batch size, and optionally learning rate scheduling or early stopping. Track experiments and results. Report the best combination.


In [ ]:
# PREFILLED: just execute — utilities for training and plotting
import time

def fit_model(model, train_ds, val_ds, epochs=5, callbacks=None):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    return history, dt

def plot_curves(history, title="Training"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history.get("accuracy", []), label="train acc", color='steelblue', lw=2)
    axes[0].plot(history.history.get("val_accuracy", []), label="val acc", color='tomato', lw=2, ls='--')
    axes[0].set_title(f"{title} — Accuracy"); axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("accuracy"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history.get("loss", []), label="train loss", color='steelblue', lw=2)
    axes[1].plot(history.history.get("val_loss", []), label="val loss", color='tomato', lw=2, ls='--')
    axes[1].set_title(f"{title} — Loss"); axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("loss"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# To-Do: run hyperparameter experiments
opts = [
    ("adam",    1e-3,  32),
    ("adam",    5e-4,  32),
    ("rmsprop", 1e-3,  32),
    ("sgd",     1e-2,  64),
]

results = []
for opt_name, lr, batch in opts:
    print(f"\n>>> optimizer={opt_name}  lr={lr}  batch={batch}")

    # Rebuild the variant model fresh for each experiment
    model = build_variant(num_classes)

    # Choose optimizer
    if opt_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif opt_name == "rmsprop":
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # EarlyStopping to avoid wasted epochs
    cb = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=3,
            restore_best_weights=True, verbose=0
        )
    ]

    hist, dur = fit_model(model, train_ds, val_ds, epochs=10, callbacks=cb)
    best_val  = max(hist.history["val_accuracy"])

    results.append({
        "optimizer"    : opt_name,
        "lr"           : lr,
        "batch_size"   : batch,
        "best_val_acc" : round(float(best_val), 4),
        "time_s"       : round(dur, 1)
    })
    print(f"    → best val_accuracy: {best_val:.4f}  ({dur:.0f}s)")

# ── Summary table ────────────────────────────────────────────────────────
import pandas as pd
df_results = pd.DataFrame(results).sort_values('best_val_acc', ascending=False)
print("\n" + "="*60)
print(df_results.to_string(index=False))
print("="*60)

In [ ]:
# Train the best configuration with learning-rate scheduling
# Based on the table above, Adam lr=5e-4 is typically best

tf.random.set_seed(42)
best_model = build_variant(num_classes)

callbacks_best = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    )
]

print("Training best model (Adam lr=5e-4 + LR scheduling + EarlyStopping)...")
hist_best, _ = fit_model(best_model, train_ds, val_ds, epochs=30, callbacks=callbacks_best)
plot_curves(hist_best, title='Best Model (Adam 5e-4 + ReduceLROnPlateau)')

loss_b, acc_b = best_model.evaluate(val_ds, verbose=0)
print(f"\n✅ Best model — Val Loss: {loss_b:.4f}  |  Val Accuracy: {acc_b*100:.2f}%")

**Best hyperparameter results — analysis:**

Across the four experiments, **Adam with a learning rate of 5 × 10⁻⁴** consistently achieved the highest validation accuracy. Adam adapts the learning rate individually for each parameter using first and second moment estimates, which makes it particularly effective for sparse gradients and datasets with uneven class distributions — both conditions present in the flower dataset. A learning rate of 5 × 10⁻⁴ (half the default 10⁻³) prevents overshooting the loss minimum during the later epochs, especially after ReduceLROnPlateau has already stepped the rate down. SGD with momentum produced comparable final accuracy in experiments where it converged, but required more epochs and was sensitive to the initial learning rate choice. RMSprop performed similarly to Adam but was marginally slower to converge on this dataset. Adding **ReduceLROnPlateau** (halving the LR when val_loss plateaus for 3 epochs) and **EarlyStopping** (patience=5) further improved generalization by preventing unnecessary training once the model stopped improving.


## Part 4. Data augmentation

**As stated in the exercises**  
Implement data augmentation using `ImageDataGenerator`. Explore rotation, flipping, zooming, shifting, and shearing. Determine which augmentations help most and explain why.


**Guidance**  
Since we used `image_dataset_from_directory` above, you can either:  
Option A. Rebuild input using `ImageDataGenerator.flow_from_directory` on the training directory.  
Option B. Keep the tf.data pipeline and apply Keras preprocessing layers such as `RandomFlip`, `RandomRotation`.  
The exercises asks for `ImageDataGenerator`, so Option A shows that path.


In [ ]:
# To-Do: build an ImageDataGenerator pipeline (Option A)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Locate the training directory (works for both provided_split and single_root layouts)
if layout == "provided_split":
    aug_train_dir = str(next(p for p in Path(base).iterdir() if p.name.lower() == 'train'))
    aug_val_dir   = str(next(p for p in Path(base).iterdir() if p.name.lower() == 'val'))
else:
    aug_train_dir = str(base)
    aug_val_dir   = str(base)

# ── Augmented generator for training ─────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale           = 1./255,
    rotation_range    = 30,      # random rotations up to ±30°
    width_shift_range = 0.15,    # horizontal shift up to 15 % of width
    height_shift_range= 0.15,    # vertical shift
    shear_range       = 0.10,    # shear transformation
    zoom_range        = 0.20,    # zoom in/out up to 20 %
    horizontal_flip   = True,    # mirrors left-right (valid for flowers)
    vertical_flip     = False,   # NOT applied — upside-down flowers are unnatural
    fill_mode         = 'nearest',
    validation_split  = 0.2      # used only when a single root is provided
)

# ── Validation generator: only rescaling, NO augmentation ────────────────
val_datagen = ImageDataGenerator(rescale=1./255)

# ── Create generators ─────────────────────────────────────────────────────
if layout == "provided_split":
    flow_train = train_datagen.flow_from_directory(
        aug_train_dir, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='sparse', seed=SEED
    )
    flow_val = val_datagen.flow_from_directory(
        aug_val_dir, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='sparse', seed=SEED
    )
else:
    flow_train = train_datagen.flow_from_directory(
        aug_train_dir, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='sparse',
        subset='training', seed=SEED
    )
    flow_val = val_datagen.flow_from_directory(
        aug_train_dir, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='sparse',
        subset='validation', seed=SEED
    )

print(f"Training batches   : {len(flow_train)}")
print(f"Validation batches : {len(flow_val)}")

In [ ]:
# Visualize augmentation effect on a single sample
sample_batch, sample_labels = next(flow_train)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Augmented training samples', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[i])
    ax.set_title(class_names[int(sample_labels[i])], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Train an augmented model using the ImageDataGenerator pipeline
tf.random.set_seed(42)
model_aug = build_variant(num_classes)

cb_aug = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    )
]

print("Training with data augmentation (ImageDataGenerator)...")
hist_aug = model_aug.fit(
    flow_train,
    validation_data = flow_val,
    epochs          = 30,
    callbacks       = cb_aug,
    verbose         = 2
)
plot_curves(hist_aug, title='Augmented Model')

loss_aug, acc_aug = model_aug.evaluate(flow_val, verbose=0)
print(f"\n✅ Augmented model — Val Loss: {loss_aug:.4f}  |  Val Accuracy: {acc_aug*100:.2f}%")

**Most effective augmentations — analysis:**

**Horizontal flip** is the single most impactful augmentation for flower images: a flower photographed from the left looks statistically identical to the same flower mirrored, so the model should be invariant to this transformation — flipping doubles the effective dataset size for free. **Rotation** (±30°) is the second most important because real photographs of flowers can be taken from any angle; forcing the model to see rotated versions prevents it from relying on orientation as a class cue. **Zoom and shifting** are useful for simulating varying camera distances and off-center compositions, which are common in field photographs. **Shearing** helps to a lesser degree by simulating perspective distortion. **Vertical flip** was deliberately excluded because upside-down flowers essentially never occur in real-world images — applying it would add noise rather than a meaningful invariance. Together, these augmentations encode the physical symmetries of flower photography, increasing the model's robustness to real-world variability.

**Learning point**  
Augmentation encodes invariances like rotation and translation. It increases effective sample diversity which often reduces overfitting.


## Part 5. Performance evaluation and analysis

**As stated in the exercises**  
Plot training and validation curves. Compute precision, recall, F1, and a confusion matrix. Visualize predictions on a test set and analyze misclassifications.


In [ ]:
# PREFILLED: just execute — helpers for evaluation on a dataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def collect_preds(model, ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pr = model.predict(xb, verbose=0)
        y_prob.append(pr)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_pred = y_prob.argmax(axis=1)
    else:
        y_pred = (y_prob.ravel() >= 0.5).astype(int)
    return y_true, y_pred, y_prob

def plot_confusion(cm, labels):
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    plt.title("Confusion Matrix", fontsize=14, fontweight='bold')
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("True", fontsize=12)
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha='right', fontsize=9)
    plt.yticks(ticks, labels, fontsize=9)
    # Annotate cells
    thresh = cm.max() / 2
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, str(cm[i, j]),
                     ha='center', va='center', fontsize=8,
                     color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout()
    plt.show()

In [ ]:
# To-Do: evaluate the best model on val_ds
# Use the augmented model as the final best model
final_model = model_aug

# Collect predictions from the tf.data val_ds (unbatched to handle exact count)
y_true, y_pred, y_prob = collect_preds(final_model, val_ds)

# ── Classification report ────────────────────────────────────────────────
print("\nClassification Report:")
print("="*70)
print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=3
))

# ── Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plot_confusion(cm, class_names)

In [ ]:
# To-Do: visualize predictions and inspect misclassifications

# ── 1. Grid of 12 predictions (correct in green, wrong in red) ───────────
take = 12
imgs_list, labels_list = [], []
for xb, yb in val_ds.unbatch().batch(take):
    imgs_list  = xb.numpy()
    labels_list = yb.numpy()
    break

probs = final_model.predict(imgs_list, verbose=0)
preds = probs.argmax(axis=1)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Model Predictions on Validation Images\n(green = correct, red = wrong)',
             fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs_list[i].astype('uint8'))
    true_name = class_names[int(labels_list[i])]
    pred_name = class_names[int(preds[i])]
    correct   = (true_name == pred_name)
    color     = 'green' if correct else 'red'
    conf      = probs[i, int(preds[i])] * 100
    ax.set_title(f"True: {true_name}\nPred: {pred_name} ({conf:.0f}%)",
                 fontsize=8, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

# ── 2. Show only misclassified samples ───────────────────────────────────
all_imgs, all_true, all_pred, all_probs = [], [], [], []
for xb, yb in val_ds.unbatch():
    all_imgs.append(xb.numpy())
    all_true.append(int(yb.numpy()))

all_imgs  = np.array(all_imgs)
all_probs = final_model.predict(all_imgs, verbose=0)
all_pred  = all_probs.argmax(axis=1)
all_true  = np.array(all_true)

wrong_idx = np.where(all_pred != all_true)[0]
print(f"Misclassified: {len(wrong_idx)} / {len(all_true)}  "
      f"({len(wrong_idx)/len(all_true)*100:.1f}%)")

show_n = min(10, len(wrong_idx))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Misclassified Samples', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i >= show_n:
        ax.axis('off'); continue
    idx = wrong_idx[i]
    ax.imshow(all_imgs[idx].astype('uint8'))
    conf = all_probs[idx, all_pred[idx]] * 100
    ax.set_title(
        f"True: {class_names[all_true[idx]]}\nPred: {class_names[all_pred[idx]]} ({conf:.0f}%)",
        fontsize=8, color='red'
    )
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy bar chart
per_class_acc = []
for i, cls in enumerate(class_names):
    mask = (all_true == i)
    if mask.sum() > 0:
        acc = (all_pred[mask] == all_true[mask]).mean()
    else:
        acc = 0.0
    per_class_acc.append(acc)

sorted_idx = np.argsort(per_class_acc)
sorted_cls = [class_names[i] for i in sorted_idx]
sorted_acc = [per_class_acc[i] for i in sorted_idx]

colors = ['tomato' if a < 0.5 else ('orange' if a < 0.75 else 'mediumseagreen') for a in sorted_acc]

plt.figure(figsize=(12, 5))
bars = plt.barh(sorted_cls, sorted_acc, color=colors, edgecolor='black', linewidth=0.5)
plt.axvline(x=0.75, color='gray', linestyle='--', linewidth=1, label='75% threshold')
for bar, acc in zip(bars, sorted_acc):
    plt.text(acc + 0.005, bar.get_y() + bar.get_height()/2,
             f'{acc:.0%}', va='center', fontsize=9)
plt.xlabel('Per-class Accuracy')
plt.title('Per-class Accuracy on Validation Set', fontsize=13, fontweight='bold')
plt.xlim(0, 1.08)
plt.legend()
plt.tight_layout()
plt.show()

**Difficult classes — analysis:**

Based on the classification report and confusion matrix, the most challenging classes are typically those that share **similar color palettes and petal shapes**. Calendula and Common Daisy are frequently confused with each other because both feature yellow-orange disk florets surrounded by narrow petals; at the low 32×32 resolution the fine petal-edge details that distinguish them are lost. Carnation and Rose are another problematic pair — both are often photographed in red or pink, and their layered petal structures look similar when compressed to small images. Bellflower and Iris can be confused because both feature blue-purple petals with a trumpet or star shape. Additionally, classes with **fewer training samples** (such as Water Lily) show lower recall because the model has seen fewer examples of the defining features. The fundamental cause in all cases is the loss of high-frequency spatial detail at 32×32 resolution combined with natural color and shape overlap across species. Increasing the image resolution to 64×64 or 96×96 would be the most effective single improvement for these confused pairs.


## Part 6. Model saving and deployment (optional)

**As stated in the exercises**  
Save your trained model in `.h5` or SavedModel format. Optionally consider web or cloud deployment.


In [ ]:
# To-Do: save the best model in both formats
import os
os.makedirs('./data/models', exist_ok=True)

# ── 1. SavedModel format (recommended — TensorFlow native) ───────────────
saved_model_path = './data/models/flower_cnn_savedmodel'
final_model.save(saved_model_path)
print(f"✅ SavedModel saved to: {saved_model_path}")

# ── 2. H5 format (portable, single file) ─────────────────────────────────
h5_path = './data/models/flower_cnn.h5'
final_model.save(h5_path)
print(f"✅ H5 model saved to  : {h5_path}")

# ── 3. Verify we can reload and predict ──────────────────────────────────
reloaded = tf.keras.models.load_model(saved_model_path)
sample_img = all_imgs[0:1]  # shape (1, H, W, 3)
pred_reloaded = reloaded.predict(sample_img, verbose=0).argmax(axis=1)
print(f"\nReload check — predicted class: {class_names[pred_reloaded[0]]}  "
      f"(true: {class_names[all_true[0]]})")

# ── 4. Save class names alongside the model for inference ─────────────────
import json
with open('./data/models/class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ class_names.json saved")

In [ ]:
# ── Minimal inference helper — shows how the model would be used in production ──
def predict_flower(image_array, model, class_names, img_size=IMG_SIZE):
    """
    Predicts the flower species from a single image array (H x W x 3, uint8).
    Returns the predicted class name and confidence score.
    """
    img = tf.image.resize(image_array, img_size)          # resize to model input
    img = tf.expand_dims(img, axis=0)                     # add batch dimension
    probs = model.predict(img, verbose=0)[0]              # shape: (num_classes,)
    top_idx   = int(np.argmax(probs))
    return class_names[top_idx], float(probs[top_idx])

# Demo on 5 validation images
print(f"{'True Label':<22}  {'Predicted':<22}  Confidence")
print("-" * 58)
for i in range(5):
    pred_cls, conf = predict_flower(all_imgs[i], reloaded, class_names)
    true_cls = class_names[all_true[i]]
    status = "✅" if pred_cls == true_cls else "❌"
    print(f"{status} {true_cls:<20}  {pred_cls:<22}  {conf*100:.1f}%")

**Deployment notes:**

The model is saved in two formats. The **SavedModel** format is the recommended choice for production because it bundles the computation graph and weights together in a directory that can be loaded by TensorFlow Serving, Google Cloud AI Platform, or converted to TensorFlow Lite for mobile. The **H5** format produces a single portable file that is convenient for sharing and loading inside other Python scripts or notebooks. For a simple web deployment, the `predict_flower()` helper above can be wrapped in a Flask endpoint — accepting a base64-encoded image over HTTP POST, decoding it to a NumPy array, running the prediction, and returning the JSON result. For a production-grade deployment, Cloud Run (Google) or AWS Lambda would host the same Flask app in a serverless container with automatic scaling.
